In [3]:
import sys
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sb
import cvxpy as cp
import math
#!{sys.executable} -m pip install yfinance
import yfinance as yf
#!{sys.executable} -m pip install pandas_datareader
from pandas_datareader import data as web

print("USING:", sys.executable)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)




#from sklearn.linear_model import ElasticNetCV
#import pandas as pd
#import numpy as np
#import datetime as dt
# from scipy import minimize
# from sklearn.linear_model import ElasticNetCV

# Import Relevant Modules
# If cvxpy is required, install it in a separate notebook cell:

#import cvxpy as cp

USING: c:\Users\User\quant39\Scripts\python.exe
numpy: 2.0.2
pandas: 2.3.3


In [17]:
tickers = [
    "AAPL",
    "MSFT",
    "JPM",
    "XOM",
    "JNJ",
    "WMT",
    "KO",
    "PG",
    "CAT",
    "IBM"
]


# Download adjusted daily stock prices.
downloaded_data = yf.download(
    tickers=tickers,
    start="1994-12-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

daily_prices = downloaded_data["Close"]
daily_prices = daily_prices.reindex(columns=tickers)


# Convert daily prices to monthly prices and returns.
monthly_prices = daily_prices.resample("ME").last()

monthly_returns = monthly_prices.pct_change(fill_method=None)
monthly_returns.index = monthly_returns.index.to_period("M")
monthly_returns = monthly_returns.loc["1995-01":"2025-12"]

print(monthly_returns.head())


Ticker       AAPL      MSFT       JPM       XOM       JNJ       WMT        KO  \
Date                                                                            
1995-01  0.035257 -0.028630  0.087107  0.028807  0.061644  0.076471  0.019417   
1995-02 -0.018981  0.061053  0.028846  0.034362 -0.018739  0.038251  0.047619   
1995-03 -0.107597  0.128969 -0.048314  0.043053  0.048458  0.081121  0.029079   
1995-04  0.085107  0.149385  0.105960  0.043152  0.092437 -0.073171  0.031042   
1995-05  0.089615  0.035932  0.104790  0.037923  0.022381  0.047369  0.060215   

Ticker         PG       CAT       IBM  
Date                                   
1995-01  0.056355 -0.063904 -0.018707  
1995-02  0.021113  0.004867  0.046835  
1995-03 -0.003759  0.075060  0.091362  
1995-04  0.060173  0.058750  0.152207  
1995-05  0.028622  0.029914 -0.014559  


In [16]:
# Read in data from K French Library
ff_data = pd.read_csv('F-F_Research_Data_5_Factors_2x3.csv')
lt_rev_data = pd.read_csv('F-F_LT_Reversal_Factor.csv')
st_rev_data = pd.read_csv('F-F_ST_Reversal_Factor.csv')
mom_data = pd.read_csv('F-F_Momentum_Factor.csv')

# Rename date column

ff_data.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
lt_rev_data.rename(columns={'Unnamed: 0': 'Date1'}, inplace=True)
st_rev_data.rename(columns={'Unnamed: 0': 'Date2'}, inplace=True)
mom_data.rename(columns={'Unnamed: 0': 'Date3'}, inplace=True)

# Combine factor returns for first dataset
combined_all = pd.concat([ff_data, lt_rev_data, st_rev_data, mom_data], axis=1)
combined_all.to_csv('combined_factors.csv', index=False)

# Remove redundant date columns and reorder
combined_all.drop(columns=['Date1', 'Date2', 'Date3'], inplace=True)

# Create factor return matrix by dropping date and creating (Mkt-Rf) as the first column
# factor_returns_only_d1 = combined_all.drop(columns=['Date', 'RF'])

# Check the data
combined_all.head()
# factor_returns_only_d1.head()

ParserError: Error tokenizing data. C error: Expected 2 fields in line 5, saw 7


In [ ]:
# Filter to get dataset 1 (1996-2005)
combined_d1 = combined_all.loc[(pd.to_numeric(combined_all['Date'], errors='coerce') >= 199601) 
                 & (pd.to_numeric(combined_all['Date'], errors='coerce') <= 200512)]
factor_returns_d1 = combined_d1.drop(columns=['Date', 'RF'])
risk_free_rate_d1 = combined_d1['RF'].astype(float) / 100

# Filter to get dataset 2 (2006-2015)
combined_d2 = combined_all.loc[(pd.to_numeric(combined_all['Date'], errors='coerce') >= 200601) 
                 & (pd.to_numeric(combined_all['Date'], errors='coerce') <= 201512)]
factor_returns_d2 = combined_d2.drop(columns=['Date', 'RF'])
risk_free_rate_d2 = combined_d2['RF'].astype(float) / 100

# Filter to get dataset 3 (2016-2025)
combined_d3 = combined_all.loc[(pd.to_numeric(combined_all['Date'], errors='coerce') >= 201601) 
                 & (pd.to_numeric(combined_all['Date'], errors='coerce') <= 202512)]
factor_returns_d3 = combined_d3.drop(columns=['Date', 'RF'])
risk_free_rate_d3 = combined_d3['RF'].astype(float) / 100

# factor_returns_d1.head()
mask = factor_returns_d1.eq("Mkt-RF")
print(mask.any())

rows, cols = np.where(mask)

for r, c in zip(rows, cols):
    print("Row:", r)
    print("Column:", factor_returns_d1.columns[c])

In [18]:
# Check filtered data for dataset 1 (1995-2015)
# combined_d1.head()
factor_returns_d1 = factor_returns_d1.astype(float)
factor_returns_d1 = factor_returns_d1 / 100
# combined_d2.head()
factor_returns_d2 = factor_returns_d2.astype(float)
factor_returns_d2 = factor_returns_d2 / 100
# combined_d3.head()
factor_returns_d3 = factor_returns_d3.astype(float)
factor_returns_d3 = factor_returns_d3 / 100
factor_returns_d1.head()


# Convert monthly returns to excess returns by subtracting the 
# risk-free rate from the monthly returns for each dataset
excess_returns_d1 = monthly_returns.loc["1996-01":"2005-12"].sub(risk_free_rate_d1.values / 100, axis=0)
excess_returns_d2 = monthly_returns.loc["2006-01":"2015-12"].sub(risk_free_rate_d2.values / 100, axis=0)
excess_returns_d3 = monthly_returns.loc["2016-01":"2025-12"].sub(risk_free_rate_d3.values / 100, axis=0)

monthly_returns_d1 = monthly_returns.loc["1996-01":"2005-12"]
monthly_returns_d2 = monthly_returns.loc["2006-01":"2010-12"]
monthly_returns_d3 = monthly_returns.loc["2016-01":"2025-12"]

excess_returns_d2.head()


NameError: name 'factor_returns_d1' is not defined

In [5]:
 #For each base model, define a function to obtain the
# factor loadings

# OLS factor loadings
def OLS_get_factor_loadings(factor_returns, excess_returns):
    """
    Parameters:
    ___________
    factor_returns: np.ndarray
        Matrix of historical factor returns of shape (T,K)
        where T is the number of periods in the sample and
        K is the number of factors considered
    ___________
    excess_returns : np.ndarray
        Matrix of historical excess asset returns of shape
        (T,N) where T is as above and N is the number of 
        assets
    """
    F = np.asarray(factor_returns); y = np.asarray(excess_returns)

    # Check dimensions are correct
    if F.ndim != 2:
        raise ValueError("The dimension of factor returns must be 2.")
    if y.ndim != 2:
        raise ValueError("The dimension of excess returns must be 2.")
    
    # Establish time periods in sample, number of assets, number of factors
    T = F.shape[0]
    N = y.shape[1]
    K = F.shape[1]

    # Check each data array has T time periods in the sample
    assert(F.shape[0] == y.shape[0])

    # Add column of ones corresponding to intercept
    X = np.column_stack((np.ones(T), F))

    # Compute OLS Solution : (X^T @ X)^{-1} @ X^T y
    factor_loadings = np.linalg.inv(X.T @ X) @ X.T @ y

    alpha = factor_loadings[0,:]
    beta = factor_loadings[1:,:]

    return alpha, beta

In [ ]:
# 3-Factor Fama-French factor loadings
def FF3_get_factor_loadings(factor_returns, excess_returns):
    """
    Parameters:
    ___________
    factor_returns: np.ndarray
        Matrix of historical factor returns of shape (T,K)
        where T is the number of periods in the sample and
        K is the number of factors considered
        It is assumed the first three columns are named
        'Mkt-Rf', 'HML', 'SMB' exactly
    ___________
    excess_returns : np.ndarray
        Matrix of historical excess asset returns of shape
        (T,N) where T is as above and N is the number of 
        assets
    """
    # Resize the factor_returns matrix to only contain 3-Factor
    # Fama-French factors: Mkt-Rf, HML, SMB
    three_factor_returns = factor_returns[:,[0,1,2]]

    # Use existing OLS method to obtain the factor loadings for
    # 3-Factor Fama-French Model
    return OLS_get_factor_loadings(three_factor_returns, excess_returns)



In [6]:
def EN_get_factor_loadings(factor_returns, excess_returns, opt_lambda_1, opt_lambda_2):
    """
    Parameters:
    ___________
    factor_returns: np.ndarray
        Matrix of historical factor returns of shape (T,K)
        where T is the number of periods in the sample and
        K is the number of factors considered
    ___________
    excess_returns : np.ndarray
        Matrix of historical excess asset returns of shape
        (T,N) where T is as above and N is the number of 
        assets
    opt_lambda : float
        Scalar value which is the optimal choice among the tested lambda
        hyperparamters for the Elastic Net strategy
    """
    # Set dataframes as arrays
    F = np.asarray(factor_returns); y = np.asarray(excess_returns)

    # Check dimensions are correct
    if F.ndim != 2:
        raise ValueError("The dimension of factor returns must be 2.")
    if y.ndim != 2:
        raise ValueError("The dimension of excess returns must be 2.")
    
    # Establish time periods in sample, number of assets, number of factors
    T = F.shape[0]
    N = y.shape[1]
    K = F.shape[1]

    # Check each data array has T time periods in the sample
    assert(F.shape[0] == y.shape[0])

    # Add column of ones corresponding to intercept
    X = np.column_stack((np.ones(T), F))

    beta = cp.Variable((K+1, N))
    objective = cp.Minimize( cp.sum_squares(X @ beta - y) + opt_lambda_1 * cp.norm1(beta[1:, :]) + opt_lambda_2 * cp.sum_squares(beta[1:, 1]) )
    problem = cp.Problem(objective)
    problem.solve()


    return beta.value[0, :] , beta.value[1:, :]





In [7]:
def get_lambdas(excess_returns, factor_returns):
    """
    Select Elastic Net lambda values using four expanding-window
    time-series validation folds.

    Parameters
    ----------
    factor_returns

    excess_returns : array-like
        Shape (60, N).

    lambda_1_values : array-like
        Candidate L1 penalty coefficients.

    lambda_2_values : array-like
        Candidate L2 penalty coefficients.

    Returns
    -------
    best_lambda_1 : float
    best_lambda_2 : float
    results : list
        Validation results for every candidate pair.
    """
    
    lambda1_values = np.linspace(0.001, 0.1, 10)  # Lambda1 values to test
    lambda2_values = np.linspace(0.001, 0.1, 10) # Lambda2 values to test

    F = np.asarray(factor_returns, dtype=float)
    Y = np.asarray(excess_returns, dtype=float)
    

    folds = [(0, 36, 36, 42), (0, 42, 42, 48), (0, 48, 48, 54), (0, 54, 54, 60)]

    best_lambda_1 = None
    best_lambda_2 = None
    best_average_rmse = np.inf

    results = []

    for lambda_1 in lambda1_values:
        for lambda_2 in lambda2_values:

            fold_rmses = []

            for (train_start, train_end, validation_start, validation_end) in folds:

                F_train = F[train_start:train_end]
                Y_train = Y[train_start:train_end]

                F_validation = F[
                    validation_start:validation_end
                ]

                Y_validation = Y[
                    validation_start:validation_end
                ]

                intercepts, loadings = EN_get_factor_loadings(F_train, Y_train,
                    lambda_1,lambda_2)

                predicted_returns = (F_validation @ loadings + intercepts)

                fold_rmse = np.sqrt(np.mean((Y_validation - predicted_returns) ** 2))
                fold_rmses.append(fold_rmse)

            average_rmse = np.mean(fold_rmses)

            results.append({"lambda_1": lambda_1,"lambda_2": lambda_2,
                            "average_rmse": average_rmse, "fold_rmses": fold_rmses})

            if average_rmse < best_average_rmse:
                best_average_rmse = average_rmse
                best_lambda_1 = lambda_1
                best_lambda_2 = lambda_2

    return best_lambda_1, best_lambda_2, results


In [8]:
def get_mean_cov(alpha, B, factor_returns, excess_returns):
    """
    Estimate the expected asset returns and asset covariance matrix.

    Parameters
    ----------
    alpha : np.ndarray
        Intercept vector of shape (N,).

    B : np.ndarray
        Factor-loading matrix of shape (K, N).

    factor_returns : np.ndarray
        Factor-return matrix of shape (T, K).

    excess_returns : np.ndarray
        Asset excess-return matrix of shape (T, N).

    Returns
    -------
    mean_vector : np.ndarray
        Expected excess-return vector of shape (N,).

    Q : np.ndarray
        Asset covariance matrix of shape (N, N).
    """

    alpha = np.asarray(alpha, dtype=float).reshape(-1)
    B = np.asarray(B, dtype=float)
    factor_returns = np.asarray(factor_returns, dtype=float)
    excess_returns = np.asarray(excess_returns, dtype=float)

    T, K = factor_returns.shape
    N = excess_returns.shape[1]

    if B.shape != (K, N):
        raise ValueError(
            f"B must have shape ({K}, {N}), "
            f"but received {B.shape}"
        )

    if alpha.shape != (N,):
        raise ValueError(
            f"alpha must have shape ({N},), "
            f"but received {alpha.shape}"
        )

    # Expected asset returns:
    # E[r] = alpha + B' E[f]
    F_bar = np.mean(factor_returns, axis=0)

    mean_vector = alpha + B.T @ F_bar

    # Factor covariance matrix, shape (K, K)
    centered_factors = factor_returns - F_bar

    F = (
        centered_factors.T
        @ centered_factors
    ) / (T - 1)

    # Predicted historical returns, shape (T, N)
    predicted_returns = alpha + factor_returns @ B

    # Regression residuals, shape (T, N)
    residuals = excess_returns - predicted_returns

    # Diagonal residual covariance
    residual_variances = np.sum(
        residuals ** 2,
        axis=0
    ) / (T - K - 1)

    E = np.diag(residual_variances)

    # Asset covariance matrix, shape (N, N)
    Q = B.T @ F @ B + E

    return mean_vector, Q


In [9]:
def get_return_prediction_error(
    alpha,
    B,
    factor_returns,
    excess_returns
):
    alpha = np.asarray(alpha, dtype=float).reshape(-1)
    B = np.asarray(B, dtype=float)
    factor_returns = np.asarray(factor_returns, dtype=float)
    excess_returns = np.asarray(excess_returns, dtype=float)

    # Predicted returns:
    # alpha has shape (N,)
    # factor_returns @ B has shape (H, N)
    predicted_returns = alpha + factor_returns @ B

    errors = excess_returns - predicted_returns

    mean_squared_error = np.mean(errors ** 2)

    return mean_squared_error

  



def get_cov_prediction_error(predicted_cov, last_excess_returns):
    """
    Calculate the RMSE between a model's predicted covariance matrix
    and the realized sample covariance matrix over the previous
    holding period.

    Parameters
    ----------
    predicted_cov : np.ndarray
        Predicted covariance matrix from the previous rebalance,
        shape (N, N).

    last_excess_returns : np.ndarray
        Excess returns over the previous holding period,
        shape (H, N).

    Returns
    -------
    float
        RMSE between the predicted and realized covariance matrices.
    """

    predicted_cov = np.asarray(predicted_cov, dtype=float)
    last_excess_returns = np.asarray(last_excess_returns, dtype=float)

    # Sample covariance matrix over the previous holding period
    realized_cov = np.cov(
        last_excess_returns,
        rowvar=False,
        ddof=1
    )

    if predicted_cov.shape != realized_cov.shape:
        raise ValueError(
            f"Shape mismatch: predicted covariance has shape "
            f"{predicted_cov.shape}, while realized covariance has "
            f"shape {realized_cov.shape}."
        )

    rmse = np.sqrt(
        np.mean((predicted_cov - realized_cov) ** 2)
    )

    return rmse


In [10]:
def MVO(mean_returns, abs_returns, cov, prev_weights, Transaction_costs):

    mean_returns = np.asarray(mean_returns, dtype=float).reshape(-1)
    abs_returns = np.asarray(abs_returns, dtype=float)
    cov = np.asarray(cov, dtype=float)
    prev_weights = np.asarray(prev_weights, dtype=float).reshape(-1)

    # Compound each asset's returns over the previous holding period.
    # abs_returns has shape (H, N), so axis=0 produces shape (N,).
    asset_growth = np.prod(1.0 + abs_returns, axis=0)

    # Value of each portfolio position after the holding period.
    drifted_position_values = prev_weights * asset_growth

    # Convert position values back into portfolio weights.
    drifted_weights = (
        drifted_position_values
        / np.sum(drifted_position_values)
    )

    n = len(mean_returns)
    costs = Transaction_costs

    target_return = np.mean(mean_returns)

    x = cp.Variable(n)

    problem = cp.Problem(
        cp.Minimize(
            0.5 * cp.quad_form(x, cov)
        ),
        [
            mean_returns @ x
            - costs * cp.sum(cp.abs(x - drifted_weights))
            >= target_return,

            cp.sum(x) == 1,
            x >= 0
        ]
    )

    problem.solve(verbose=False)

    if x.value is None:
        raise ValueError(
            f"MVO optimization failed. Status: {problem.status}"
        )

    return np.asarray(x.value).reshape(-1)

In [11]:
# Define quick helper functions

def drift_weights(weights, asset_returns):
    ending_values = weights * (1.0 + asset_returns)
    return ending_values / ending_values.sum()

def calculate_turnover(new_weights, current_weights):
    return 0.5 * np.sum(np.abs(new_weights - current_weights))

In [12]:
def rebalancing(train_factor_returns, train_excess_returns, last_factor_returns, 
                last_excess_returns, last_abs_returns, prev_weights, Transaction_costs, 
                Temperature_M, Temperature_Q, beta_OLS_prev, beta_FF3_prev, beta_EN_prev,
                alpha_OLS_prev, alpha_FF3_prev, alpha_EN_prev,
                cov_OLS_prev, cov_FF3_prev, cov_EN_prev, is_first,
                equal_weight, OLS_test, FF3_test, EN_test):
    """
    The rebalancing function performs the portfolio rebalancing at each rebalancing point. 
    """

    alpha_OLS , beta_OLS = OLS_get_factor_loadings(train_factor_returns, train_excess_returns)
    alpha_FF3 , beta_FF3 = FF3_get_factor_loadings(train_factor_returns, train_excess_returns)
    opt_lambda1 , opt_lambda2, _ = get_lambdas(train_excess_returns, train_factor_returns)
    alpha_EN , beta_EN = EN_get_factor_loadings(train_factor_returns, train_excess_returns, opt_lambda1, opt_lambda2)

    mean_OLS , cov_OLS = get_mean_cov(alpha_OLS , beta_OLS, train_factor_returns, train_excess_returns)
    three_factor_returns = train_factor_returns[:,[0,1,2]]
    mean_FF3 , cov_FF3 = get_mean_cov(alpha_FF3, beta_FF3, three_factor_returns, train_excess_returns)
    mean_EN , cov_EN = get_mean_cov(alpha_EN, beta_EN, train_factor_returns, train_excess_returns)

    # Calculate the prediction error across all monthly returns for all assets
    # in the period since the last rebalancing, for each base model
    # Note: consider just using error vs. mean return vector, not all return points
    if ( (not is_first) and (not equal_weight) and (not OLS_test)
        and (not FF3_test) and (not EN_test) ):
        OLS_prev_Merror = get_return_prediction_error(
            alpha_OLS_prev, beta_OLS_prev, last_factor_returns, last_excess_returns)
        FF3_prev_Merror = get_return_prediction_error(
            alpha_FF3_prev, beta_FF3_prev, last_factor_returns[:,[0,1,2]], last_excess_returns)
        EN_prev_Merror = get_return_prediction_error(
            alpha_EN_prev, beta_EN_prev, last_factor_returns, last_excess_returns)

        # Calculate the prediction error across all monthly returns for all assets
        # in the period since the last rebalancing, for each base model
    
        OLS_prev_Qerror = get_cov_prediction_error(cov_OLS_prev, last_excess_returns)
        FF3_prev_Qerror = get_cov_prediction_error(cov_FF3_prev, last_excess_returns)
        EN_prev_Qerror = get_cov_prediction_error(cov_EN_prev, last_excess_returns)

        # Re-allocate based on the errors of the base models over the previous period
        # in predicting the excess returns, ensuring weights add to 1
        epsilon = 1e-8 # To protect against NaN errors
        

        recip_exp_mean = 1 / (math.exp(-1 * OLS_prev_Merror * Temperature_M)
                              + math.exp(-1 * FF3_prev_Merror * Temperature_M)
                              + math.exp(-1 * EN_prev_Merror * Temperature_M))
        
        OLS_mean_weight = math.exp(-1 * Temperature_M * OLS_prev_Merror) * recip_exp_mean
        FF3_mean_weight = math.exp(-1 * Temperature_M * FF3_prev_Merror) * recip_exp_mean
        EN_mean_weight = math.exp(-1 * Temperature_M * EN_prev_Merror) * recip_exp_mean

        # Re-allocate based on the errors of the base models over the previous period in
        # predicting the covariance of the excess returns, ensuring weights add to 1
        recip_exp_Q = 1 / (math.exp(-1 * OLS_prev_Qerror * Temperature_Q)
                              + math.exp(-1 * FF3_prev_Qerror * Temperature_Q)
                              + math.exp(-1 * EN_prev_Qerror * Temperature_Q))
        
        OLS_cov_weight = math.exp(-1 * Temperature_Q * OLS_prev_Qerror) * recip_exp_Q
        FF3_cov_weight = math.exp(-1 * Temperature_Q * FF3_prev_Qerror) * recip_exp_Q
        EN_cov_weight = math.exp(-1 * Temperature_Q * EN_prev_Qerror) * recip_exp_Q
    
    elif OLS_test:
        OLS_mean_weight = 1
        FF3_mean_weight = 0
        EN_mean_weight = 0

        OLS_cov_weight = 1
        FF3_cov_weight = 0
        EN_cov_weight = 0

    elif FF3_test:
        OLS_mean_weight = 0
        FF3_mean_weight = 1
        EN_mean_weight = 0

        OLS_cov_weight = 0
        FF3_cov_weight = 1
        EN_cov_weight = 0

    elif EN_test:
        OLS_mean_weight = 0
        FF3_mean_weight = 0
        EN_mean_weight = 1

        OLS_cov_weight = 0
        FF3_cov_weight = 0
        EN_cov_weight = 1

    else:
        # Set weights equal in first rebalancing (when no previous error available)
        OLS_mean_weight = 1/3
        FF3_mean_weight = 1/3
        EN_mean_weight = 1/3

        OLS_cov_weight = 1/3
        FF3_cov_weight = 1/3
        EN_cov_weight = 1/3

    # Get the ensemble estimate for the mean return vector
    ensemble_mean = (
        OLS_mean_weight * mean_OLS + FF3_mean_weight * mean_FF3
    + EN_mean_weight * mean_EN
    )

    # Get the ensemble estimate for the covariance matrix
    ensemble_cov = (
        OLS_cov_weight * cov_OLS + FF3_cov_weight * cov_FF3
    + EN_cov_weight * cov_EN
    )
    # Perform MVO on the asset portfolio using the ensemble mean and covarinace
    # estimates
    alloc_ensemble = MVO(ensemble_mean, last_abs_returns, ensemble_cov,
                         prev_weights,  Transaction_costs)

    alloc_OLS = MVO(mean_OLS, last_abs_returns, cov_OLS, prev_weights, Transaction_costs)
    alloc_FF3 = MVO(mean_FF3, last_abs_returns, cov_FF3, prev_weights, Transaction_costs)
    alloc_EN = MVO(mean_EN, last_abs_returns, cov_EN, prev_weights, Transaction_costs)

    # Return the ensemble allocation and the factor loadings for each of
    # the base models (for future MSE calculation)
    return (
    alloc_ensemble,
    alpha_OLS,
    beta_OLS,
    alpha_FF3,
    beta_FF3,
    alpha_EN,
    beta_EN,
    cov_OLS,
    cov_FF3,
    cov_EN,
    OLS_mean_weight,
    FF3_mean_weight,
    EN_mean_weight,
    OLS_cov_weight,
    FF3_cov_weight,
    EN_cov_weight
)


In [13]:
def run_ensemble_backtest(holding_period, training_window, trans_cost_rate, 
                          all_factor_returns, all_excess_returns, Temperature_M,
                          Temperature_Q,
                          all_abs_returns, initial_value, risk_free_rate,
                          equal_weight, OLS_test, FF3_test, EN_test):
    """ 
    """
    portfolio_value = float(initial_value) # Initial portfolio value

    factor_returns = np.asarray(all_factor_returns, dtype=float)
    excess_returns = np.asarray(all_excess_returns, dtype=float)
    abs_returns = np.asarray(all_abs_returns, dtype=float)

    N = excess_returns.shape[1]
    total_time = factor_returns.shape[0]

    H = holding_period
    T = training_window
    c = trans_cost_rate

    current_weights = np.repeat(1.0 / N, N)
    alpha_OLS_prev = None
    alpha_FF3_prev = None
    alpha_EN_prev = None

    beta_OLS_prev = None
    beta_FF3_prev = None
    beta_EN_prev = None

    cov_OLS_prev = None
    cov_FF3_prev = None
    cov_EN_prev = None

    is_first = True

    portfolio_value_history = [initial_value]
    portfolio_return_history = []
    weight_history = []
    turnover_history = []
    transaction_cost_history = []
    rebalance_history = []
    model_Mweight_history = []
    model_Qweight_history = []

    rebalance_indices = set(range(T,total_time, H))

    for time in range(total_time):
        
        if (time in rebalance_indices):
            
            ( new_weights , alpha_OLS , beta_OLS , alpha_FF3 , beta_FF3 , 
             alpha_EN, beta_EN ,
             cov_OLS , cov_FF3 , cov_EN ,
             OLS_Mweight, FF3_Mweight, EN_Mweight,
             OLS_Qweight, FF3_Qweight, EN_Qweight) = rebalancing(
                        factor_returns[time - T:time, :], excess_returns[time - T:time, :],
                        factor_returns[time - H:time, :], excess_returns[time - H:time, :],
                        abs_returns[time - H:time, :],
                        current_weights, trans_cost_rate, Temperature_M,
                        Temperature_Q,
                        beta_OLS_prev, beta_FF3_prev, beta_EN_prev,
                        alpha_OLS_prev, alpha_FF3_prev, alpha_EN_prev,
                        cov_OLS_prev, cov_FF3_prev, cov_EN_prev, is_first,
                         equal_weight, OLS_test, FF3_test, EN_test)
            
            turnover = calculate_turnover(new_weights, current_weights)
            # This is the fraction of the portfolio traded

            cost_fraction = c * turnover * 2 # To account for same fee for buying and selling
            dollar_cost = portfolio_value * cost_fraction
            portfolio_value -= dollar_cost

            current_weights = new_weights.copy()

            # Update histories
            turnover_history.append(turnover)
            transaction_cost_history.append(dollar_cost)
            rebalance_history.append(time)

            alpha_OLS_prev = alpha_OLS
            beta_OLS_prev = beta_OLS
            alpha_FF3_prev = alpha_FF3
            beta_FF3_prev = beta_FF3
            alpha_EN_prev = alpha_EN
            beta_EN_prev = beta_EN

            cov_OLS_prev = cov_OLS
            cov_FF3_prev = cov_FF3
            cov_EN_prev = cov_EN

            is_first = False

        if time >= training_window + 1:
            current_asset_returns = abs_returns[time, :]
            monthly_portfolio_return = float(current_weights @ current_asset_returns)

            portfolio_value *= (1.0 + monthly_portfolio_return)
            portfolio_return_history.append(monthly_portfolio_return)
            portfolio_value_history.append(portfolio_value)

            current_weights = drift_weights(current_weights, current_asset_returns)
            weight_history.append(np.round(current_weights.copy(),4))
            average_turnover = np.mean(turnover_history)

            model_Mweight_history.append(np.asarray([round(OLS_Mweight,4), round(FF3_Mweight,4), round(EN_Mweight,4)]))
            model_Qweight_history.append(np.asarray([round(OLS_Qweight,4), round(FF3_Qweight,4), round(EN_Qweight,4)]))

    
    sharpe_ratio = ( (np.mean(portfolio_return_history) - np.mean(risk_free_rate))
                        / pow(np.var(portfolio_return_history), 0.5) )
        
    
    plt.figure(figsize=(10,6))
    x = np.linspace(training_window, total_time, total_time-training_window)
    plt.plot(x, portfolio_value_history)
    plt.title("Portfolio Value")
    plt.xlabel("Month")
    plt.ylabel("$")
    plt.grid(True)

    return {"final_portfolio_value": portfolio_value,
            "portfolio_value_history": np.asarray(portfolio_value_history),
            "weight_history": np.asarray(weight_history),
            "turnover_history": np.asarray(turnover_history),
            "average_turnover": average_turnover,
            "transaction_cost_history": np.asarray(transaction_cost_history),
            "rebalance_indices": rebalance_history,
            "sharpe_ratio:": sharpe_ratio,
            "model_Mweight_history": model_Mweight_history,
            "model_Qweight_history": model_Qweight_history}

In [15]:
# Cell that runs the backtest

Temperature_M = 0 # We set the temperature of the softmax function to provide a balance of
        # stability and flexibility in how the weights are allocated
Temperature_Q = 0
holding_period = 6
# total_time = 120
initial_value = 100000
training_window = 60
trans_cost_rate = 0.0025


In [ ]:
all_factor_returns = factor_returns_d1
all_excess_returns = excess_returns_d1
all_abs_returns = monthly_returns_d1

risk_free_rate = risk_free_rate_d1
